# HalluGuard × MiniCheck — 1,100-row sample experiment

This notebook runs a deterministic preliminary experiment over **100 test rows per dataset × 11 LLM-AggreFact datasets**. HalluGuard is loaded as `MiniCheck(model_name="HalluGuard-Qwen3-4B", ...)`, and every prediction is produced by `MiniCheck.score()` with `chunk_size=500`.

No custom chunker, direct HalluGuard inference loop, threshold tuning, comparison model, or fabricated failure prediction is used. Results, checkpoints, metrics, and the environment/config manifest are written under `/kaggle/working/halluguard_minicheck_sample`.


## 1. Clean-session setup and MiniCheck integration


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# vLLM 0.10.2 V1 can enter Torch Inductor CUDAGraph paths even with
# enforce_eager=True; a long T4 run reproduced an allocator-state crash.
# V0 plus eager execution avoids that subsystem. Attention stays auto-selected.
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORKING = Path("/kaggle/working")
if not WORKING.exists():
    WORKING = Path.cwd()
SOURCE_DIR = WORKING / "minicheck_halluguard_source"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"

def run_checked(args, cwd=None):
    print("+", " ".join(map(str, args)))
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True, text=True)

gpu_before = subprocess.run(
    ["nvidia-smi"], text=True, capture_output=True, check=True
).stdout
print(gpu_before)
assert "T4" in gpu_before, "This notebook targets Kaggle T4 GPUs."

if SOURCE_DIR.exists():
    shutil.rmtree(SOURCE_DIR)
run_checked([
    "git", "clone", "--filter=blob:none", "https://github.com/Liyan06/MiniCheck.git", str(SOURCE_DIR)
])
run_checked(["git", "checkout", MINICHECK_COMMIT], cwd=SOURCE_DIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "vllm==0.10.2", "transformers==4.57.6", "datasets", "nltk", "scikit-learn",
    "pandas", "huggingface_hub", "accelerate"
])

# Read-only credential handling. Never place token values in notebook source or output.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN is available from the environment/Kaggle Secrets (value not printed).")
else:
    print("HF_TOKEN is not set; continuing because the referenced public assets may not require it.")


In [ ]:
import ast
from pathlib import Path

mini_path = SOURCE_DIR / "minicheck" / "minicheck.py"
inf_path = SOURCE_DIR / "minicheck" / "inference.py"
mini = mini_path.read_text(encoding="utf-8")
inf = inf_path.read_text(encoding="utf-8")

def replace_exact(text, old, new, label, expected=1):
    found = text.count(old)
    if found != expected:
        raise RuntimeError(f"{label}: expected {expected} anchor(s), found {found}")
    return text.replace(old, new, expected)

allowed_tail = "'Granite-Guardian-3.3-8B']"
mini = replace_exact(
    mini,
    allowed_tail,
    "'Granite-Guardian-3.3-8B', 'HalluGuard-Qwen3-4B']",
    "MiniCheck allowed-model list",
    expected=2,
)

anchor = """        elif model_name == 'Granite-Guardian-3.3-8B':
            if not max_tokens or max_tokens<2048:
                print(\"For Granite Guardian 3.3 - fixing the max_tokens to be 2048\")
                max_tokens=2048

            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
halluguard_constructor = """        elif model_name == 'HalluGuard-Qwen3-4B':
            if not max_tokens or max_tokens < 2:
                raise ValueError(\"HalluGuard max_tokens must be at least 2.\")
            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )
"""
mini = replace_exact(
    mini, anchor, halluguard_constructor + anchor,
    "MiniCheck HalluGuard constructor",
)

model_anchor = "        if model_id == 'Bespoke-MiniCheck-7B':\n"
inf = replace_exact(
    inf,
    model_anchor,
    "        self.model_revision = None\n        self.vllm_kwargs = {}\n\n" + model_anchor,
    "LLMCheck defaults",
)

anchor = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        else:
            raise ValueError(\"model_id must be 'Bespoke-MiniCheck-7B'\")"""
replacement = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        elif model_id == 'HalluGuard-Qwen3-4B':
            self.model_id = 'lrsbrgrn/HalluGuard-Qwen3-4B'
            self.operating_mode = \"halluguard_binary\"
            self.model_revision = '2bf3858f8bd0c22b0c410f6b0c855098727a2554'
            self.vllm_kwargs = {
                'enforce_eager': True,
                'disable_custom_all_reduce': True,
                'gpu_memory_utilization': 0.80,
                'max_num_seqs': 4,
            }
        else:
            raise ValueError(\"Unsupported LLMCheck model_id: %s\" % model_id)"""
inf = replace_exact(inf, anchor, replacement, "LLMCheck model dispatch")

inf = replace_exact(
    inf,
    "            model=self.model_id, ",
    "            model=self.model_id,\n            revision=self.model_revision, ",
    "vLLM revision",
)
inf = replace_exact(
    inf,
    "            enable_prefix_caching=self.enable_prefix_caching",
    "            enable_prefix_caching=self.enable_prefix_caching,\n            **self.vllm_kwargs",
    "vLLM HalluGuard stability kwargs",
)

anchor = """        self.sampling_params = SamplingParams(
            temperature=0,
            max_tokens=self.max_tokens,
            stop_token_ids=terminators,
            logprobs=5
        )"""
replacement = """        if self.operating_mode == \"halluguard_binary\":
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=[self.tokenizer.eos_token_id],
                seed=42,
            )
        else:
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                logprobs=5
            )
        self.last_halluguard_records = []"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard sampling parameters")

anchor = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        return text"""
replacement = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        elif self.operating_mode == \"halluguard_binary\":
            prompt = (
                \"Document:\\n\" + doc + \"\\n\\nClaim:\\n\" + claim
                + \"\\n\\nReply exactly YES if the claim is fully supported by the document. \"
                + \"Reply exactly NO if the claim is contradicted or unsupported by the document. \"
                + \"No explanation, reasoning, punctuation, XML, or extra text.\"
            )
            text = self.tokenizer.apply_chat_template(
                [{\"role\": \"user\", \"content\": prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        return text"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard chat template")

inf = replace_exact(
    inf,
    "class LLMCheck:\n",
    "class HalluGuardParseError(ValueError):\n    pass\n\n\nclass LLMCheck:\n",
    "HalluGuard parse exception",
)

parser_anchor = '    def get_support_prob_hybrid_gg(self, response, marker="score"):\n'
parser = """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == \"YES\":
            return \"GROUNDED\", 1.0, final_text
        if final_text == \"NO\":
            return \"HALLUCINATED\", 0.0, final_text
        raise HalluGuardParseError(
            \"HalluGuard binary output must be exactly trimmed uppercase YES or NO. \"
            + \"Raw output: \" + repr(response_text[:4000])
        )

    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({
            \"classification\": label,
            \"support_score\": score,
            \"raw_output\": raw_text,
        })
        return score

"""
inf = replace_exact(
    inf, parser_anchor, parser + parser_anchor,
    "HalluGuard strict parser",
)

anchor = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]"""
replacement = anchor + """
        elif self.operating_mode == \"halluguard_binary\":
            self.last_halluguard_records = []
            probs_per_chunk_sentence = [
                self.get_support_score_halluguard(response) for response in responses
            ]"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard response dispatch")

ast.parse(mini, filename=str(mini_path))
ast.parse(inf, filename=str(inf_path))
mini_path.write_text(mini, encoding="utf-8")
inf_path.write_text(inf, encoding="utf-8")
print("Patched pinned MiniCheck with deterministic HalluGuard binary support.")

run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(SOURCE_DIR)])

import nltk
for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)

sys.path.insert(0, str(SOURCE_DIR))
from minicheck.minicheck import MiniCheck
from minicheck.inference import HalluGuardParseError
print("MiniCheck import path:", __import__("minicheck").__file__)


In [ ]:
import json
import platform
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score

MODEL_NAME = "HalluGuard-Qwen3-4B"
MODEL_ID = "lrsbrgrn/HalluGuard-Qwen3-4B"
MODEL_REVISION = "2bf3858f8bd0c22b0c410f6b0c855098727a2554"
DATASET_ID = "lytang/LLM-AggreFact"
SPLIT = "test"
CHUNK_SIZE = 500
MAX_MODEL_LEN = 4096
MAX_TOKENS = 8
TENSOR_PARALLEL_SIZE = 1
ENABLE_PREFIX_CACHING = False
RANDOM_STATE = 42
CACHE_DIR = str(WORKING / "hf_cache")

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

OUTPUT_DIR = WORKING / "halluguard_minicheck_sample"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Deterministic 100-per-dataset test sample


In [ ]:
SAMPLE_PER_DATASET = 100
SUB_BATCH_SIZE = 4
CHECKPOINT_PATH = OUTPUT_DIR / "predictions_checkpoint.csv"
FINAL_PATH = OUTPUT_DIR / "predictions_final.csv"
METRICS_PATH = OUTPUT_DIR / "dataset_metrics.csv"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
RETRY_FAILED_ON_RESUME = True
MAX_ATTEMPTS = 2
RESUME_SOURCE = None

raw = load_dataset(DATASET_ID, split=SPLIT, token=hf_token)
full_df = raw.to_pandas()
full_df["source_index"] = np.arange(len(full_df), dtype=np.int64)
dataset_names = sorted(full_df["dataset"].unique().tolist())
assert len(dataset_names) == 11, f"Expected 11 datasets, found {len(dataset_names)}: {dataset_names}"

sample_parts = []
for dataset_name, group in full_df.groupby("dataset", sort=True):
    assert len(group) >= SAMPLE_PER_DATASET, f"{dataset_name} has fewer than 100 test rows"
    part = group.sample(n=SAMPLE_PER_DATASET, random_state=RANDOM_STATE).copy()
    sample_parts.append(part.sort_values("source_index"))
eval_df = pd.concat(sample_parts, ignore_index=True)
eval_df["sample_id"] = eval_df["dataset"].astype(str) + "::" + eval_df["source_index"].astype(str)
assert len(eval_df) == 1100 and eval_df["sample_id"].is_unique
eval_df[["sample_id", "dataset", "source_index", "label"]].to_csv(
    OUTPUT_DIR / "sample_index.csv", index=False
)

# Resume across Kaggle sessions by attaching a prior notebook output as Input.
input_root = Path("/kaggle/input")
if not CHECKPOINT_PATH.exists() and input_root.exists():
    inspected = []
    for candidate in sorted(input_root.rglob("predictions_checkpoint.csv")):
        try:
            frame = pd.read_csv(candidate)
            required = {"sample_id", "source_index", "dataset", "label", "status"}
            if required.issubset(frame.columns):
                inspected.append((int((frame["status"] == "OK").sum()), len(frame), candidate, frame))
        except Exception as exc:
            print(f"Ignoring unreadable checkpoint {candidate}: {type(exc).__name__}: {exc}")
    if inspected:
        _, _, selected, imported = max(inspected, key=lambda item: (item[0], item[1], str(item[2])))
        shutil.copy2(selected, OUTPUT_DIR / "imported_checkpoint_original.csv")
        recovered = imported.loc[imported["status"] == "OK"].copy()
        valid_keys = eval_df[["sample_id", "source_index", "dataset", "label"]]
        recovered = recovered.merge(
            valid_keys, on=["sample_id", "source_index", "dataset", "label"], how="inner"
        )
        sort_col = "attempt_count" if "attempt_count" in recovered.columns else "source_index"
        recovered = recovered.sort_values(sort_col).drop_duplicates("sample_id", keep="last")
        recovered.to_csv(CHECKPOINT_PATH, index=False)
        RESUME_SOURCE = str(selected)
        print(f"Recovered {len(recovered)} successful rows from {selected}; failures will be retried.")

display(eval_df.groupby("dataset")["label"].agg(["count", "sum"]))


## 3. Load HalluGuard through MiniCheck


In [ ]:
scorer = MiniCheck(
    model_name="HalluGuard-Qwen3-4B",
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    max_tokens=MAX_TOKENS,
    enable_prefix_caching=ENABLE_PREFIX_CACHING,
    cache_dir=CACHE_DIR,
)
assert scorer.model.operating_mode == "halluguard_binary"
assert scorer.model.model_id == MODEL_ID
assert scorer.model.model_revision == MODEL_REVISION
assert scorer.model.vllm_kwargs == {
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "gpu_memory_utilization": 0.80,
    "max_num_seqs": 4,
}
print("Loaded HalluGuard through MiniCheck on", TENSOR_PARALLEL_SIZE, "GPUs.")


## 4. Checkpoint/resume helpers

    Small batches are attempted first. If one fails, each row is retried independently through `MiniCheck.score()` so parsing and inference failures can be attributed precisely. Failed rows have null prediction/score values; no fallback label is invented.


In [ ]:
RESULT_COLUMNS = [
    "sample_id", "dataset", "source_index", "label", "prediction", "support_score",
    "n_chunks", "chunk_sentence_shape", "status", "error_type", "error_message",
    "raw_model_output", "elapsed_seconds", "attempt_count",
]

def empty_results():
    return pd.DataFrame(columns=RESULT_COLUMNS)

def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return empty_results()
    checkpoint = pd.read_csv(CHECKPOINT_PATH)
    if "raw_model_output" not in checkpoint.columns:
        checkpoint["raw_model_output"] = None
    missing = set(RESULT_COLUMNS) - set(checkpoint.columns)
    if missing:
        raise ValueError(f"Checkpoint schema mismatch; missing columns: {sorted(missing)}")
    checkpoint = checkpoint[checkpoint["sample_id"].isin(eval_df["sample_id"])].copy()
    checkpoint["attempt_count"] = checkpoint["attempt_count"].fillna(0).astype(int)
    checkpoint = checkpoint.sort_values("attempt_count").drop_duplicates("sample_id", keep="last")
    print(f"Loaded {len(checkpoint)} checkpoint rows from {CHECKPOINT_PATH}")
    return checkpoint

def atomic_save(frame, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def exception_chain(exc):
    seen = set()
    while exc is not None and id(exc) not in seen:
        seen.add(id(exc))
        yield exc
        exc = exc.__cause__ or exc.__context__

def error_status(exc):
    return "PARSE_FAILURE" if any(
        isinstance(item, HalluGuardParseError) for item in exception_chain(exc)
    ) else "INFERENCE_FAILURE"

def is_engine_dead(exc):
    for item in exception_chain(exc):
        name = type(item).__name__.lower()
        message = str(item).lower()
        if "enginedead" in name or "engine core" in message and "dead" in message:
            return True
    return False

def successful_records(batch, preds, probs, chunks, matrices, elapsed, prior_attempts):
    per_row_elapsed = elapsed / max(len(batch), 1)
    rows = []
    for pos, row in enumerate(batch.itertuples(index=False)):
        rows.append({
            "sample_id": row.sample_id, "dataset": row.dataset,
            "source_index": int(row.source_index), "label": int(row.label),
            "prediction": int(preds[pos]), "support_score": float(probs[pos]),
            "n_chunks": len(chunks[pos]),
            "chunk_sentence_shape": str(np.asarray(matrices[pos]).shape),
            "status": "OK", "error_type": None, "error_message": None,
            "raw_model_output": None, "elapsed_seconds": per_row_elapsed,
            "attempt_count": int(prior_attempts.get(row.sample_id, 0)) + 1,
        })
    return rows

def failure_record(row, exc, prior_attempts, elapsed):
    status = error_status(exc)
    message = str(exc)[:8000]
    return {
        "sample_id": row["sample_id"], "dataset": row["dataset"],
        "source_index": int(row["source_index"]), "label": int(row["label"]),
        "prediction": None, "support_score": None, "n_chunks": None,
        "chunk_sentence_shape": None, "status": status,
        "error_type": type(exc).__name__, "error_message": message,
        "raw_model_output": message if status == "PARSE_FAILURE" else None,
        "elapsed_seconds": elapsed,
        "attempt_count": int(prior_attempts.get(row["sample_id"], 0)) + 1,
    }

def score_batch(batch, prior_attempts):
    started = time.perf_counter()
    preds, probs, chunks, matrices = scorer.score(
        docs=batch["doc"].astype(str).tolist(),
        claims=batch["claim"].astype(str).tolist(),
        chunk_size=CHUNK_SIZE,
    )
    return successful_records(
        batch, preds, probs, chunks, matrices,
        time.perf_counter() - started, prior_attempts,
    )

def score_individually(batch, prior_attempts):
    rows = []
    for _, row in batch.iterrows():
        started = time.perf_counter()
        try:
            rows.extend(score_batch(row.to_frame().T, prior_attempts))
        except Exception as exc:
            rows.append(failure_record(
                row, exc, prior_attempts, time.perf_counter() - started
            ))
            if is_engine_dead(exc):
                return rows, True
    return rows, False


## 5. Run with checkpointing


In [ ]:
results = load_checkpoint()
prior_attempts = {
    row.sample_id: int(row.attempt_count) for row in results.itertuples(index=False)
}
done_ids = set(results.loc[
    (results["status"] == "OK") | (results["attempt_count"] >= MAX_ATTEMPTS),
    "sample_id",
]) if RETRY_FAILED_ON_RESUME else set(results["sample_id"])

pending = eval_df.loc[~eval_df["sample_id"].isin(done_ids)].copy()
print(f"Checkpoint rows: {len(results)} | pending rows: {len(pending)}")
run_started = time.perf_counter()
ENGINE_ABORTED = False

for start in range(0, len(pending), SUB_BATCH_SIZE):
    batch = pending.iloc[start:start + SUB_BATCH_SIZE]
    fatal_engine = False
    try:
        new_rows = score_batch(batch, prior_attempts)
    except Exception as batch_exc:
        print(f"Batch {start}:{start + len(batch)} failed ({type(batch_exc).__name__}); isolating rows.")
        if is_engine_dead(batch_exc):
            first = batch.iloc[0]
            new_rows = [failure_record(first, batch_exc, prior_attempts, 0.0)]
            fatal_engine = True
        else:
            new_rows, fatal_engine = score_individually(batch, prior_attempts)

    new_df = pd.DataFrame(new_rows, columns=RESULT_COLUMNS)
    results = pd.concat([results, new_df], ignore_index=True)
    results = results.sort_values("attempt_count").drop_duplicates("sample_id", keep="last")
    results = results.sort_values(["dataset", "source_index"]).reset_index(drop=True)
    atomic_save(results, CHECKPOINT_PATH)
    for record in new_rows:
        prior_attempts[record["sample_id"]] = int(record["attempt_count"])

    print(
        f"saved {len(results)}/1100 | OK={int((results.status == 'OK').sum())} "
        f"| parse_failures={int((results.status == 'PARSE_FAILURE').sum())} "
        f"| inference_failures={int((results.status == 'INFERENCE_FAILURE').sum())}"
    )
    if fatal_engine:
        ENGINE_ABORTED = True
        (OUTPUT_DIR / "engine_abort.json").write_text(
            json.dumps(new_rows[-1], indent=2, default=str), encoding="utf-8"
        )
        print("vLLM engine died; checkpoint saved and remaining rows were left NOT_RUN.")
        break

run_elapsed = time.perf_counter() - run_started
print(f"Run segment finished in {run_elapsed / 60:.1f} minutes. Checkpoint: {CHECKPOINT_PATH}")


## 6. Preliminary metrics and final artifacts

    Balanced Accuracy is computed only over successful predictions. Coverage and failure counts are reported alongside every metric, making incomplete results explicit. Macro BAcc is the unweighted mean of the 11 per-dataset BAcc values; no threshold is tuned.


In [ ]:
results = load_checkpoint()
merged = eval_df.merge(
    results,
    on=["sample_id", "dataset", "source_index", "label"],
    how="left",
    validate="one_to_one",
)
merged["status"] = merged["status"].fillna("NOT_RUN")

metric_rows = []
for dataset_name, group in merged.groupby("dataset", sort=True):
    valid = group[group["status"] == "OK"].copy()
    bacc = None
    if len(valid) and valid["label"].nunique() == 2:
        bacc = float(balanced_accuracy_score(
            valid["label"].astype(int), valid["prediction"].astype(int)
        ))
    metric_rows.append({
        "dataset": dataset_name,
        "sampled_rows": len(group),
        "valid_predictions": len(valid),
        "coverage": len(valid) / len(group),
        "parse_failures": int((group["status"] == "PARSE_FAILURE").sum()),
        "inference_failures": int((group["status"] == "INFERENCE_FAILURE").sum()),
        "not_run": int((group["status"] == "NOT_RUN").sum()),
        "balanced_accuracy": bacc,
    })

metrics = pd.DataFrame(metric_rows)
macro_bacc = float(metrics["balanced_accuracy"].mean()) if metrics["balanced_accuracy"].notna().any() else None
metrics["macro_balanced_accuracy"] = macro_bacc
atomic_save(metrics, METRICS_PATH)
atomic_save(merged, FINAL_PATH)
display(metrics)
print("Macro Balanced Accuracy:", macro_bacc)

try:
    import vllm
    vllm_version = vllm.__version__
except Exception:
    vllm_version = None

gpu_snapshot = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=True).stdout
(OUTPUT_DIR / "nvidia_smi_final.txt").write_text(gpu_snapshot, encoding="utf-8")
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "HalluGuard through MiniCheck, deterministic 100-per-dataset sample",
    "minicheck_commit": MINICHECK_COMMIT,
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset": DATASET_ID,
    "split": SPLIT,
    "dataset_rows_total": len(full_df),
    "datasets": dataset_names,
    "sample_per_dataset": SAMPLE_PER_DATASET,
    "sample_rows": len(eval_df),
    "source_index_preserved": True,
    "random_state": RANDOM_STATE,
    "chunk_size": CHUNK_SIZE,
    "max_model_len": MAX_MODEL_LEN,
    "max_tokens": MAX_TOKENS,
    "generation": {"enable_thinking": False, "temperature": 0, "max_tokens": 8, "format": "exact YES/NO", "seed": 42},
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "enable_prefix_caching": ENABLE_PREFIX_CACHING,
    "vllm_engine": {
        "version": "0.10.2",
        "engine_mode": "V0",
        "enforce_eager": True,
        "disable_custom_all_reduce": True,
        "gpu_memory_utilization": 0.80,
        "max_num_seqs": 4,
        "reason": "Single-T4 eager-mode stability after observed tensor-parallel engine failures",
    },
    "threshold": "> 0.5 (MiniCheck unchanged)",
    "metric": "Balanced Accuracy; unweighted macro across datasets",
    "sub_batch_size": SUB_BATCH_SIZE,
    "valid_predictions": int((merged["status"] == "OK").sum()),
    "parse_failures": int((merged["status"] == "PARSE_FAILURE").sum()),
    "inference_failures": int((merged["status"] == "INFERENCE_FAILURE").sum()),
    "not_run": int((merged["status"] == "NOT_RUN").sum()),
    "macro_balanced_accuracy": macro_bacc,
    "resume_source": RESUME_SOURCE,
    "engine_aborted": ENGINE_ABORTED,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "vllm": vllm_version,
    "cuda": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "artifacts": {
        "checkpoint": str(CHECKPOINT_PATH),
        "final_predictions": str(FINAL_PATH),
        "dataset_metrics": str(METRICS_PATH),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

if manifest["not_run"]:
    print("The run is incomplete. Re-run this notebook to resume from the checkpoint.")
else:
    print("All sampled rows were attempted. Failures, if any, remain explicit and have no prediction.")
